In [ ]:
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

In [ ]:
# ---- Sheet config ----
SPREADSHEET_ID = "1JFpAxDlJhM6UhL3u5ZuFOoIGvlf5AuG7qNhrRygIL3k"
WORKSHEET_NAME = "Community_Survey"

TARGET_COLS = [
    "Q2A - If you are dissatisfied with any of the items listed in Quality of Life, why?",
    "Q4A - If you are dissatisfied with any of the items listed in Economic Opportunity and Affordability, why?",
    "Q6A - If you are dissatisfied with any of the items listed in Health and Environment, why?",
    "Q8A - If you are dissatisfied with any of the items listed in Safety, why?",
    "Q10A - If you are dissatisfied with any of the items listed in Mobility, why?",
    "Q12A - If you are dissatisfied with any of the items listed in Culture and Lifelong Learning, why?",
    "Q14A - If you are dissatisfied with any of the items listed in Government that Works for All, why?",
    "Q25 - If there was one thing you could share with the Mayor regarding the City of Austin (any comment, suggestion, etc.), what would it be?",
]

RANDOM_SEED = 42  # reproducible shuffle


def _is_nonempty(x) -> bool:
    if x is None:
        return False
    s = str(x).strip()
    return s != "" and s.lower() != "nan"


def load_sheet_to_long_df(spreadsheet_id: str, worksheet_name: str, target_cols: list[str]) -> pd.DataFrame:
    # auth
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    sh = gc.open_by_key(spreadsheet_id)
    ws = sh.worksheet(worksheet_name)

    records = ws.get_all_records()
    df = pd.DataFrame(records)

    # validate headers
    missing = [c for c in target_cols if c not in df.columns]
    if missing:
        raise KeyError(
            "Missing expected column headers (exact match required):\n- " + "\n- ".join(missing)
        )

    # preserve original sheet row number (row 1 header; get_all_records starts at row 2)
    df = df.copy()
    df["source_row"] = df.index + 2

    long_parts = []
    for col in target_cols:
        part = df[["source_row", col]].copy()
        part = part.rename(columns={col: "comment_text"})
        part["question_col"] = col
        part["comment_text"] = part["comment_text"].astype(str).map(lambda s: s.strip())
        part = part[part["comment_text"].map(_is_nonempty)]
        long_parts.append(part)

    long_df = pd.concat(long_parts, ignore_index=True)

    # shuffle reproducibly
    long_df = long_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return long_df


long_df = load_sheet_to_long_df(SPREADSHEET_ID, WORKSHEET_NAME, TARGET_COLS)
print("Unpivoted non-empty rows:", len(long_df))
print(long_df.head(3))

Unpivoted non-empty rows: 13542
   source_row                                       comment_text  \
0        7385  All Austin residents deserve safety, cultural ...   
1        9060  Taxes will require me to move from city.  Stop...   
2        6470  I feel unsafe to drive and for sure to walk by...   

                                        question_col  
0  Q25 - If there was one thing you could share w...  
1  Q14A - If you are dissatisfied with any of the...  
2  Q8A - If you are dissatisfied with any of the ...  


In [ ]:
import numpy as np

comments = long_df["comment_text"].dropna().astype(str)

# Basic counts
total = len(long_df)
non_empty = len(comments)
empty = total - non_empty

# Word counts
word_counts = comments.str.split().str.len()

# Character counts
char_counts = comments.str.len()

# Unique comments and duplicates
n_unique = comments.nunique()
n_dupes = non_empty - n_unique

# Top repeated comments
top_dupes = comments.value_counts().head(20)

# Distribution by question column
q_dist = long_df.groupby("question_col")["comment_text"].count().sort_values(ascending=False)

# Sample comments at different lengths
short = comments[word_counts <= 5].sample(min(10, (word_counts <= 5).sum()), random_state=42)
medium = comments[(word_counts > 5) & (word_counts <= 20)].sample(min(10, ((word_counts > 5) & (word_counts <= 20)).sum()), random_state=42)
long = comments[word_counts > 20].sample(min(10, (word_counts > 20).sum()), random_state=42)

print("=" * 60)
print("COMMENT TEXT PROFILE")
print("=" * 60)

print(f"\n--- Counts ---")
print(f"Total rows:          {total:,}")
print(f"Non-empty comments:  {non_empty:,}")
print(f"Empty/NaN:           {empty:,}")
print(f"Unique comments:     {n_unique:,}")
print(f"Duplicates:          {n_dupes:,}")

print(f"\n--- Word Count Stats ---")
print(f"Mean:    {word_counts.mean():.1f}")
print(f"Median:  {word_counts.median():.1f}")
print(f"Std:     {word_counts.std():.1f}")
print(f"Min:     {word_counts.min()}")
print(f"Max:     {word_counts.max()}")

print(f"\n--- Word Count Distribution ---")
bins = [0, 3, 5, 10, 20, 50, 100, float("inf")]
labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51-100", "100+"]
binned = pd.cut(word_counts, bins=bins, labels=labels)
dist = binned.value_counts().sort_index()
for label, count in dist.items():
    pct = count / non_empty * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:>7s}: {count:>5,} ({pct:5.1f}%) {bar}")

print(f"\n--- Character Count Stats ---")
print(f"Mean:    {char_counts.mean():.1f}")
print(f"Median:  {char_counts.median():.1f}")
print(f"Max:     {char_counts.max()}")

print(f"\n--- Comments Per Question ---")
for q, cnt in q_dist.items():
    short_q = q[:80] + "..." if len(q) > 80 else q
    print(f"  {cnt:>5,}  {short_q}")

print(f"\n--- Top 20 Repeated Comments ---")
for txt, cnt in top_dupes.items():
    if cnt < 2:
        break
    print(f"  {cnt:>4}x  {txt[:90]}")

print(f"\n--- Sample SHORT Comments (≤5 words) ---")
for t in short.values:
    print(f"  • {t}")

print(f"\n--- Sample MEDIUM Comments (6-20 words) ---")
for t in medium.values:
    print(f"  • {t}")

print(f"\n--- Sample LONG Comments (>20 words) ---")
for t in long.values:
    print(f"  • {t[:150]}")

COMMENT TEXT PROFILE

--- Counts ---
Total rows:          13,542
Non-empty comments:  13,542
Empty/NaN:           0
Unique comments:     13,300
Duplicates:          242

--- Word Count Stats ---
Mean:    22.6
Median:  14.0
Std:     31.0
Min:     1
Max:     1315

--- Word Count Distribution ---
      1-3:   794 (  5.9%) ██
      4-5: 1,211 (  8.9%) ████
     6-10: 2,945 ( 21.7%) ██████████
    11-20: 4,084 ( 30.2%) ███████████████
    21-50: 3,285 ( 24.3%) ████████████
   51-100:   901 (  6.7%) ███
     100+:   322 (  2.4%) █

--- Character Count Stats ---
Mean:    130.5
Median:  84.0
Max:     7421

--- Comments Per Question ---
  2,933  Q25 - If there was one thing you could share with the Mayor regarding the City o...
  2,340  Q4A - If you are dissatisfied with any of the items listed in Economic Opportuni...
  1,993  Q10A - If you are dissatisfied with any of the items listed in Mobility, why?
  1,673  Q2A - If you are dissatisfied with any of the items listed in Quality of Life, w..

In [ ]:
!pip install bertopic sentence-transformers hdbscan umap-learn

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
# --- Preprocessing ---
def clean_comment(text):
    text = text.strip()
    if text.isupper():
        text = text.capitalize()
    text = re.sub(r"\s+", " ", text)
    return text

docs = long_df["comment_text"].astype(str).map(clean_comment).tolist()

# --- Embedding model ---
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# --- UMAP: tuned for this dataset ---
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# --- HDBSCAN: larger min_cluster_size for cleaner initial clusters ---
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

# --- Vectorizer: remove noise words common in survey responses ---
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

# --- BERTopic ---
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    calculate_probabilities=True,
    verbose=True
)

# Fit
topics, probs = topic_model.fit_transform(docs)

# --- Reduce outliers ---
new_topics = topic_model.reduce_outliers(docs, topics, strategy="distributions")
topic_model.update_topics(docs, topics=new_topics)

# --- Merge to ~50 topics ---
topic_model.reduce_topics(docs, nr_topics=50)

# --- Map back to dataframe (use model's current topics post-reduction) ---
current_topics = topic_model.topics_

df_results = long_df.copy()
df_results["Topic_Number"] = current_topics
df_results["Topic_Probability"] = [max(p) for p in probs]

topic_info = topic_model.get_topic_info()
topic_label_map = dict(zip(topic_info["Topic"], topic_info["Name"]))
df_results["Topic_Label"] = df_results["Topic_Number"].map(topic_label_map)

# --- Summary ---
n_outliers = (df_results["Topic_Number"] == -1).sum()
print(f"Total comments:  {len(df_results):,}")
print(f"Topics found:    {len(topic_info) - 1}")
print(f"Outliers:        {n_outliers:,} ({n_outliers/len(df_results)*100:.1f}%)")
print(f"\n{topic_info[['Topic', 'Count', 'Name']].to_string(index=False)}")

# Preview
df_results[["comment_text", "question_col", "Topic_Number", "Topic_Label", "Topic_Probability"]].head(20)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-05 05:37:16,238 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/424 [00:00<?, ?it/s]

2026-03-05 05:37:19,970 - BERTopic - Embedding - Completed ✓
2026-03-05 05:37:19,971 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-05 05:37:33,737 - BERTopic - Dimensionality - Completed ✓
2026-03-05 05:37:33,738 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-05 05:37:36,591 - BERTopic - Cluster - Completed ✓
2026-03-05 05:37:36,596 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-05 05:37:36,990 - BERTopic - Representation - Completed ✓
100%|██████████| 3/3 [00:00<00:00,  7.33it/s]
2026-03-05 05:37:37,713 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-03-05 05:37:37,953 - BERTopic - Topic reduction - Reducing n

Total comments:  13,542
Topics found:    49
Outliers:        10 (0.1%)

 Topic  Count                                            Name
    -1     10                 -1_menace_lawless_coyotes_unity
     0   2572                              0_austin_to_the_in
     1   1078                1_housing_affordable_cost_living
     2    620                        2_city_council_the_mayor
     3    508                    3_traffic_congestion_flow_is
     4    473                  4_downtown_night_safe_homeless
     5    447             5_growth_infrastructure_city_zoning
     6    358                         6_lanes_bike_bikes_lane
     7    353                   7_water_wastewater_rates_high
     8    348            8_homeless_problem_homelessness_help
     9    347         9_transportation_public_transit_options
    10    306                    10_parks_park_parking_trails
    11    302                      11_property_taxes_tax_high
    12    279              12_library_libraries_museums_book

,comment_text,question_col,Topic_Number,Topic_Label,Topic_Probability
0,"All Austin residents deserve safety, cultural ...",Q25 - If there was one thing you could share w...,0,0_austin_to_the_in,0.563707
1,Taxes will require me to move from city. Stop...,Q14A - If you are dissatisfied with any of the...,2,2_city_council_the_mayor,0.053498
2,I feel unsafe to drive and for sure to walk by...,Q8A - If you are dissatisfied with any of the ...,4,4_downtown_night_safe_homeless,0.302157
3,"The city is very liberal, does not provide nic...",Q2A - If you are dissatisfied with any of the ...,10,10_parks_park_parking_trails,0.231517
4,Don't think they city provides enough series t...,Q14A - If you are dissatisfied with any of the...,0,0_austin_to_the_in,0.212686
5,PRESERVE SMALL LOCAL BUSINESSES. KEEP AUSTIN L...,Q25 - If there was one thing you could share w...,0,0_austin_to_the_in,0.461649
6,TOO MUCH DEMOCRACY. TOO MUCH RED TAPE-PERMITTI...,Q4A - If you are dissatisfied with any of the ...,21,21_permitting_process_permit_inspection,1.000000
7,taxes are too high and traffic is horrendous,Q4A - If you are dissatisfied with any of the ...,40,40_taxes_property_traffic_high,1.000000
8,"Because, none of these questions reflect the r...",Q8A - If you are dissatisfied with any of the ...,29,29_service_customer_you_answer,0.039243
9,LACK OF AFFORDABLE HOUSING STRANGLES,Q4A - If you are dissatisfied with any of the ...,1,1_housing_affordable_cost_living,1.000000


In [ ]:
current_topics = topic_model.topics_
idxs = [i for i, x in enumerate(current_topics) if x == 0]
samples = random.sample([docs[i] for i in idxs], 15)
for s in samples:
    print(f"  • {s[:120]}")

  • City council tried to ramrod a halfway house for tdjc inmates onto my street a number of years ago. the proposal was def
  • Preserve our history so that future generations have the opportunities to learn form it rather than bowing to one ethnic
  • The parks are disgusting. I can't. Ring my kids there any longer. The homeless are a real problem. Stop encouraging Aust
  • Austin is growing at rapid pace it is important to lift all citizens not just those with deep pockets.
  • Austin is very expensive to live, housing cost is very high
  • The overwhelming amount of traffic almost makes life in Austin dissatisfying.
  • Stop robin hood program so I am not taxed out of my home. Stop overpaying city employees & hiring more so they can do le
  • Austin is not set up for growth. Bad roads and lack of mass transit.
  • The city of austin is not a good place to work. do something about the corrupt managers that are hiring at the city.
  • Heavy traffic is the worst aspect of Austin. The 

In [ ]:
# Label topic 0 honestly, and clean up any other weak labels
topic_model.set_topic_labels({
    0: "General / Multi-Issue Comments",
    46: "Traffic & Construction"
})

# Run the dense summary to do a final sanity check
from collections import Counter
import random
random.seed(42)

current_topics = topic_model.topics_

stop = set(["the","and","to","of","a","in","is","it","that","for","are","on",
            "with","was","as","at","be","this","have","from","or","an","but",
            "not","by","if","they","their","there","than","been","has","its",
            "do","did","my","i","we","you","our","me","your","would","should",
            "can","could","will","all","no","so","too","very","just","more",
            "about","up","out","what","which","who","how","when","where","why",
            "also","had","were","them","those","these","any","each","other",
            "into","some","being","he","she","her","his","him","don","doesn",
            "didn","isn","aren","wasn","won","wouldn","couldn","shouldn","t","s",
            "re","ve","ll","m","d","am","need","get","like","much","many",
            "one","think","city","austin","feel","make","even","going","go",
            "us","been","want","really","lot","thing","things","way","people",
            "because","still","well","here","live","know","good","needs","new",
            "see","keep","take","come","better","every","been","over","most",
            "only","back","than","same","while","said","does","done","able"])

for t in sorted(set(current_topics)):
    if t == -1:
        continue
    idxs = [i for i, x in enumerate(current_topics) if x == t]
    texts = [docs[i] for i in idxs]

    words = []
    for txt in texts:
        words.extend([w.lower().strip(".,!?;:()\"'") for w in txt.split() if len(w) > 2])
    words = [w for w in words if w not in stop and w != ""]
    top5 = [w for w, _ in Counter(words).most_common(5)]

    samples = random.sample(texts, min(4, len(texts)))
    samples = [s[:100].replace("\n", " ") for s in samples]

    print(f"T{t:>2} ({len(idxs):>4}) [{', '.join(top5)}]")
    for s in samples:
        print(f"       {s}")

T 0 (2572) [traffic, housing, homeless, taxes, don't]
       Do not let big business, extremely wealth people and the state dictate what is done for Austin. List
       I think austin is expensive so that affects our ability to retire here.
       Need to mow right of ways in S. Austin more than once a year
       Don't pay companies to move to Austin.
T 1 (1078) [housing, affordable, cost, living, high]
       Overall affordability is crucial - taxes, electric and water service.
       Though there are many excellent pockets of development, there appears to be little unified planning 
       I am transitioning careers, but despite my training and active efforts to find a job, I have been un
       The apparent rush to build a condo on every available foot of ground is disturbing to me, and the de
T 2 ( 620) [council, taxes, mayor, services, money]
       The city waste too much tax payers money
       equality in service provision for all neighbors
       Shelter is another example of

In [ ]:
# topic visualization
topic_model.visualize_topics()

# Bar chart of top words per topic
topic_model.visualize_barchart(top_n_topics=15)

# See how topics relate to each other
topic_model.visualize_heatmap()

In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# Prepare export dataframe
export_df = df_results[["source_row", "question_col", "comment_text", "Topic_Number", "Topic_Label", "Topic_Probability"]].copy()
export_df["Topic_Probability"] = export_df["Topic_Probability"].round(4)

# Convert to list of lists for gspread
header = export_df.columns.tolist()
rows = export_df.values.tolist()

# Create or replace worksheet
sh = gc.open_by_key(SPREADSHEET_ID)
try:
    ws = sh.worksheet("BERTopic_Results")
    sh.del_worksheet(ws)
except gspread.exceptions.WorksheetNotFound:
    pass

ws = sh.add_worksheet(title="BERTopic_Results", rows=len(rows)+1, cols=len(header))
ws.update([header] + rows)

print(f"Wrote {len(rows):,} rows to 'BERTopic_Results' worksheet")

Wrote 13,542 rows to 'BERTopic_Results' worksheet


In [ ]:
# Save model to drive
from google.colab import drive
drive.mount('/content/drive')

# Save to Drive
topic_model.save("/content/drive/MyDrive/bertopic_austin_survey", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

print("Model saved")

Mounted at /content/drive
Model saved


In [ ]:
#topic_model = BERTopic.load("/content/drive/MyDrive/bertopic_austin_survey", embedding_model=embedding_model)

# AI Citation Section

Cleaned up formatting of print statements, Sonnet 4.6, Anthropic

Typeahead used in some cases, Google Colab - Gemini, Google